In [2]:
import sys
import os

ROOT = os.path.abspath("..")  # adjust if needed
print(ROOT)
OPENPCDET_PATH = '/storage/project/r-gchou3-0/spanse30/OpenPCDet'
PTT_PATH = os.path.join(ROOT, "PTT")
print(OPENPCDET_PATH)
print(PTT_PATH)
# # Put OpenPCDet FIRST
sys.path.insert(0, OPENPCDET_PATH)

# # Import OpenPCDet pcdet
import pcdet
print("Using pcdet from:", pcdet.__file__)

/storage/scratch1/9/spanse30
/storage/project/r-gchou3-0/spanse30/OpenPCDet
/storage/scratch1/9/spanse30/PTT
Using pcdet from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/pcdet/__init__.py


In [3]:
import pickle as pkl

In [4]:
import os
from pathlib import Path
import copy
import numpy as np
import torch
    

In [5]:
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.models import build_network, load_data_to_gpu
from pcdet.datasets import build_dataloader
from pcdet.utils import common_utils
import random

import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

import importlib
import helpers_ptt
importlib.reload(helpers_ptt)
import torch.nn.functional as F

from collections import defaultdict

In [6]:
from pcdet.config import cfg, cfg_from_yaml_file
cfg.clear()

CFG_FILE_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml' 
CKPT_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth'  # <- put your ckpt here

logger = common_utils.create_logger()
logger.info(f'Loaded cfg from {CFG_FILE_MF}')

cfg_from_yaml_file(
    CFG_FILE_MF,
    cfg
)

cfg.TAG = 'centerpoint_multiframe'
cfg.EXP_GROUP_PATH = 'waymo_multiframe'

dataset_mf, test_loader_mf, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger,
    training=False
)

len_test = len(dataset_mf)
logger.info(f'Test set length: {len_test}')
cp_mf_model = build_network(
    model_cfg=cfg.MODEL,
    num_class=len(cfg.CLASS_NAMES),
    dataset=dataset_mf
)

logger.info(f'Loading checkpoint from: {CKPT_MF}')
cp_mf_model.load_params_from_file(filename=CKPT_MF, logger=logger, to_cpu=False)
cp_mf_model.cuda()
cp_mf_model.eval()

data_iter_mf = iter(test_loader_mf)
batch_dict_mf = next(data_iter_mf)

load_data_to_gpu(batch_dict_mf)

2026-03-04 17:58:01,171   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml
2026-03-04 17:58:01,193   INFO  Loading Waymo dataset
2026-03-04 17:58:04,652   INFO  Total skipped info 0
2026-03-04 17:58:04,653   INFO  Total samples for Waymo dataset: 39987
2026-03-04 17:58:04,654   INFO  Test set length: 39987
2026-03-04 17:58:04,904   INFO  Loading checkpoint from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth
2026-03-04 17:58:04,907   INFO  ==> Loading parameters from checkpoint /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth to GPU
2026-03-04 17:58:05,151   INFO  ==> Checkpoint trained from version: pcdet+0.6.0+233f849
2026-03-04 17:58:05,161   INFO  ==> Done (loaded 288/288)


In [7]:
len(dataset_mf)

39987

In [8]:
dataset_mf[50]['points'].shape

(502083, 6)

In [9]:
import sys
for k in list(sys.modules.keys()):
    if k.startswith("pcdet"):
        del sys.modules[k]
sys.path = [p for p in sys.path if "OpenPCDet" not in p]
sys.path.insert(0, PTT_PATH)

import pcdet
print("Now using:", pcdet.__file__)

from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils

Now using: /storage/scratch1/9/spanse30/PTT/pcdet/__init__.py


In [10]:
from pcdet.config import cfg, cfg_from_yaml_file
cfg.clear()


CFG_FILE = 'tools/cfgs/waymo_models/ptt_32frames.yaml' 
CKPT = 'output/cfgs/waymo_models/default/checkpoint_epoch_6.pth'  # <- put your ckpt here

cfg_from_yaml_file(CFG_FILE, cfg)
cfg.TAG = Path(CFG_FILE).stem
cfg.EXP_GROUP_PATH = 'centerpoint_waymo_demo'

import os
# cfg.DATA_CONFIG.DATA_PATH = os.path.join(OPENPCDET_PATH, 'data', 'waymo')

logger = common_utils.create_logger()
logger.info(f'Loaded cfg from {CFG_FILE}')


dataset, test_loader, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger,
    training=False
)

len_test = len(dataset)
logger.info(f'Test set length: {len_test}')
model = build_network(
    model_cfg=cfg.MODEL,
    num_class=len(cfg.CLASS_NAMES),
    dataset=dataset
)

logger.info(f'Loading checkpoint from: {CKPT}')
model.load_params_from_file(filename=CKPT, logger=logger, to_cpu=False)
model.cuda()
model.eval()

data_iter = iter(test_loader)
batch_dict = next(data_iter)

load_data_to_gpu(batch_dict)



2026-03-04 17:58:12,800   INFO  Loaded cfg from tools/cfgs/waymo_models/ptt_32frames.yaml
2026-03-04 17:58:12,800   INFO  Loaded cfg from tools/cfgs/waymo_models/ptt_32frames.yaml
2026-03-04 17:58:12,803   INFO  Loading Waymo dataset
2026-03-04 17:58:12,803   INFO  Loading Waymo dataset
2026-03-04 17:58:15,783   INFO  Total skipped info 0
2026-03-04 17:58:15,783   INFO  Total skipped info 0
2026-03-04 17:58:15,784   INFO  Total samples for Waymo dataset: 38597
2026-03-04 17:58:15,784   INFO  Total samples for Waymo dataset: 38597
2026-03-04 17:58:15,784   INFO  Loading and reorganizing pred_boxes to dict from path: /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/eval/epoch_36/val/default/result_fixed.pkl
2026-03-04 17:58:15,784   INFO  Loading and reorganizing pred_boxes to dict from path: /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/eval/epoch_36/val/default/r

In [15]:
dataset[0]['roi_boxes'].shape

(32, 88, 9)

In [10]:
# import pickle
# pred_dir = 'experiments/segment_preds_e7'

# segment_preds = {}
# unique_segment_names = dataset.seq_name_to_infos.keys()
# for segment_name in unique_segment_names:
#     with open(f"{pred_dir}/{segment_name}_p.pkl", "rb") as f:
#         segment_preds[segment_name] = pickle.load(f)
# infos = dataset.infos

# final_preds = []

# segment_frame_counters = {}

# for info in infos:
#     segment = info['point_cloud']['lidar_sequence']
    
#     if segment not in segment_frame_counters:
#         segment_frame_counters[segment] = 0
    
#     idx = segment_frame_counters[segment]
    
#     final_preds.append(segment_preds[segment][idx])
    
#     segment_frame_counters[segment] += 1

In [11]:
# for pred, info in zip(final_preds, dataset.infos):
#     assert pred['frame_id'] == info['frame_id']

In [12]:
# import pickle
# dataset_dir = 'experiments/segment_datasets_e7'

# segment_datasets = {}
# unique_segment_names = dataset.seq_name_to_infos.keys()
# counter = 0
# for segment_name in unique_segment_names:
#     with open(f"{dataset_dir}/{segment_name}_d.pkl", "rb") as f:
#         segment_annos_dict = pickle.load(f)
#         # segment_annos = segment_annos_dict[segment_name]
        
# #         segment_annos_dict_cleaned = {}
# #         annos_cleaned = []
#         for datasample in segment_annos_dict:
#             del datasample['spoof_points']
#             del datasample['frame_points']
#             del datasample['frame_poses']
#             del datasample['frame_gt_boxes']
#         # segment_datasets[segment_name] = pickle.load(f)
#         segment_datasets[segment_name] = segment_annos_dict
#     #     counter+=1
#     # if counter == 1:
#     #     break
# infos = dataset.infos

In [13]:
# for info in dataset.infos:

#         segment = info['point_cloud']['lidar_sequence']

In [14]:
# %%capture cap
# import os
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # suppress TF C++ logs
# import logging
# logging.getLogger().setLevel(logging.ERROR)
# from copy import deepcopy

# window_results = []   # store metrics per datasample

# for segment_name, annos in segment_datasets.items():

#     for datasample in annos:

#         start, end = datasample['frame_window']

#         det_window = []
#         infos_window = []

#         segment_frame_counters = {}

#         # --- build window subset ---
#         for info in dataset.infos:

#             segment = info['point_cloud']['lidar_sequence']

#             if segment not in segment_frame_counters:
#                 segment_frame_counters[segment] = 0

#             idx = segment_frame_counters[segment]

#             if segment == segment_name and start <= idx <= end:
#                 det_window.append(segment_preds[segment][idx])
#                 infos_window.append(info)

#             segment_frame_counters[segment] += 1

#         # Safety checks
#         assert len(det_window) == len(infos_window)
#         assert len(det_window) == 32  # each datasample window is 32 frames

#         # --- Evaluate this window ---
#         dataset_subset = deepcopy(dataset)
#         dataset_subset.infos = infos_window

#         result_str, result_dict = dataset_subset.evaluation(
#             det_window,
#             cfg.CLASS_NAMES,
#             eval_metric=cfg.MODEL.POST_PROCESSING.EVAL_METRIC
#         )

#         window_results.append(result_dict)
# print(result_dict["OBJECT_TYPE_TYPE_VEHICLE_LEVEL_1/AP"])
# print(result_dict["OBJECT_TYPE_TYPE_VEHICLE_LEVEL_1/APH"])

import pickle
with open(f"clean.pkl", "rb") as f:
    clean = pickle.load(f)
    
len(clean)

38597

In [15]:
type(clean)

list

In [16]:
type(clean[0])

dict

In [18]:
# windows_frames = {}   # {segment: set(frame_idx)}

# for segment, annos in segment_datasets.items():
#     windows_frames[segment] = set()

#     for anno in annos:
#         start, end = anno['frame_window']
#         for f in range(start, end + 1):
#             windows_frames[segment].add(f)
# det_annos_subset = []
# infos_subset = []

# segment_frame_counters = {}

# for info in dataset.infos:
#     segment = info['point_cloud']['lidar_sequence']

#     if segment not in segment_frame_counters:
#         segment_frame_counters[segment] = 0

#     idx = segment_frame_counters[segment]

#     # choose which mask to use
#     if segment in windows_frames and idx in windows_frames[segment]:
#         det_annos_subset.append(segment_preds[segment][idx])
#         infos_subset.append(info)

#     segment_frame_counters[segment] += 1


In [19]:
# windows_frames['segment-10203656353524179475_7625_000_7645_000_with_camera_labels']

In [22]:
# len(infos_subset)

In [23]:
# len(segment_datasets)

In [25]:
# segment_datasets['segment-10203656353524179475_7625_000_7645_000_with_camera_labels'][0].keys()

In [26]:
elevations = np.load('waymo_top_lidar_inclinations.npy')
stats = np.load("range_conditioned_stats.npy", allow_pickle=True).item()

In [27]:
def center_trace(selected_trace):
    centered_points_trace = selected_trace['points'][:, :2] - selected_trace['box'][:2]
    angle = 0
    if(selected_trace['box'][0]<0):
        # print("rear object")
        angle = np.pi

    c = np.cos(angle)
    s = np.sin(angle)
    R = np.array([[c, -s],
                  [s, c]])


    rotated_trace = centered_points_trace @ R.T
    return rotated_trace

In [28]:
def isolate_frame_points(pts, lag):

    # 2. Force a copy to ensure memory is contiguous
    pts_clean = np.array(pts, copy=True)
    # print(pts_clean.shape)
    # 3. Create a strict mask on the last column (index 5)
    # Using a gap-based threshold (0.05) since lags are 0, 0.1, 0.2, 0.3
    lags = np.round(pts_clean[:, -1], decimals=1)
    # print(len(lags))
    target_lag = 0.1*lag
    # mask = np.abs(pts_clean[:, 5]) < 0.05
    mask = (lags == target_lag)
    # print(mask)
    # 4. Apply
    current_frame = pts_clean[mask]
    # print(f"Verified Max Lag: {current_frame[:, 5].max()}")
    return current_frame

In [29]:
dataset_mf[0]['points']

array([[ 1.7414988e+01,  1.4261880e+00,  2.0478405e-02,  2.4775203e-02,
         9.9540465e-02,  0.0000000e+00],
       [ 4.5734344e+00, -7.4932456e+00, -1.8790076e-02,  5.1468153e-02,
         4.0987249e-02,  0.0000000e+00],
       [ 1.1235543e+01, -2.5508628e+00, -2.5089548e-03,  4.2454932e-02,
         5.2697893e-02,  0.0000000e+00],
       ...,
       [ 2.4111765e+01,  1.2452555e+01,  2.4041314e-02,  1.3548975e-02,
         1.5223835e-01,  0.0000000e+00],
       [ 1.4745117e+01, -1.3711914e+01, -7.2715759e-02,  1.7872380e-01,
         2.0493625e-01,  0.0000000e+00],
       [ 2.3498015e+00,  8.1752110e+00, -8.0015995e-02,  2.3189202e-02,
         1.7565964e-02,  0.0000000e+00]], dtype=float32)

In [30]:
print(isolate_frame_points(dataset_mf[0]['points'], 0).shape[0])

504399


In [31]:
print(isolate_frame_points(dataset_mf[1]['points'], 1).shape[0])

379485


In [32]:
print(isolate_frame_points(dataset_mf[2]['points'], 2).shape[0])

251276


In [33]:
print(isolate_frame_points(dataset_mf[3]['points'], 3).shape[0])

125059


In [34]:
np.mean(dataset_mf[1]['points'][:, -1])

0.07474944

In [35]:


def spoof_frame(spf_cur, points):
    global elevations, stats
    # --- Ground anchoring correction ---

    # Use scene XYZ
    scene_xyz = points[:, :3]

    # Estimate spoof object footprint center
    spoof_xy = spf_cur[:, :2]
    center_xy = spoof_xy.mean(axis=0)

    # Select scene points near spoof footprint (radius 1.5–2.0m works well)
    radius = 2.0
    dists = np.linalg.norm(scene_xyz[:, :2] - center_xy, axis=1)
    local_mask = dists < radius

    if np.sum(local_mask) > 20:  # ensure enough support
        local_points = scene_xyz[local_mask]

        # Estimate ground as low percentile (robust against cars/buildings)
        ground_z = np.percentile(local_points[:, 2], 5)

        # Estimate spoof object's lowest point
        spoof_bottom_z = np.min(spf_cur[:, 2])

        # Compute vertical shift
        dz = ground_z - spoof_bottom_z

        # Apply rigid vertical correction
        spf_cur[:, 2] += dz
        
    #empirically sample intensities and elongations 
    r_avg = helpers_ptt.avg_spoof_range(spf_cur)
    band_stats = helpers_ptt.get_stats_for_range(r_avg, stats)
    P_two = 0.01
    if (band_stats is not None):
        print("updating intensity and elongation with empirical values")
        I = np.random.lognormal(
                band_stats["log_mu"],
                band_stats["log_sigma"],
                size=spf_cur.shape[0]
            )
        I = np.clip(I, band_stats["intensity_p1"], band_stats["intensity_p99"])
        E = np.random.choice(band_stats["elong_pool"], size=spf_cur.shape[0])
        spf_cur[:, 3] = I
        spf_cur[:, 4] = E
        
    dtheta = np.deg2rad(0.1358)

    print(f"avg spoof range: {r_avg}")

    # --- Convert to spherical ---
    r_trace, theta_trace, phi_trace = helpers_ptt.cart_2_spherical(spf_cur)
    scene_xyz = points[:, :3]
    r_scene, theta_scene, phi_scene = helpers_ptt.cart_2_spherical(scene_xyz)

    # --- Discretize scene points ---
    az_scene = np.mod(theta_scene, 2*np.pi)
    az_idx_scene = np.floor(az_scene / dtheta).astype(np.int32)
    beam_idx_scene = np.abs(elevations[:, None] - phi_scene).argmin(axis=0)

    # --- Sort scene by (az_idx, beam_idx, range) ---
    order = np.lexsort((r_scene, beam_idx_scene, az_idx_scene))
    az_idx_scene = az_idx_scene[order]
    beam_idx_scene = beam_idx_scene[order]
    r_scene = r_scene[order]
    scene_indices_sorted = order

    # Identify group boundaries
    keys = az_idx_scene * 1000 + beam_idx_scene  # unique key
    unique_keys, group_start = np.unique(keys, return_index=True)
    group_end = np.r_[group_start[1:], len(keys)]

    # Build fast lookup map from key → slice
    key_to_slice = {
        k: (start, end)
        for k, start, end in zip(unique_keys, group_start, group_end)
    }

    remove_spoof = []
    remove_scene = []

    # --- Process spoof points ---
    for idx, (r, theta, phi) in enumerate(zip(r_trace, theta_trace, phi_trace)):
        az = theta % (2*np.pi)
        az_idx = int(np.floor(az / dtheta))
        beam_idx = np.argmin(np.abs(elevations - phi))
        key = az_idx * 1000 + beam_idx

        if key not in key_to_slice:
            continue

        start, end = key_to_slice[key]
        r_group = r_scene[start:end]
        idx_group = scene_indices_sorted[start:end]

        if len(r_group) == 0:
            continue

        # Case 1: spoof behind first real return
        if r_group[0] < r:
            remove_spoof.append(idx)
        
        # Case 2: spoof in front of all
        elif r < r_group[0]:

            # Spoof becomes first return.
            # Decide if second return survives.
            if np.random.rand() < P_two:
                # Allow second return: keep nearest scene return as second
                if len(idx_group) > 0:
                    remove_scene.extend(idx_group[1:].tolist())
            else:
                # Only spoof survives
                remove_scene.extend(idx_group.tolist())

        # Case 3: spoof between returns
        else:
            k = np.searchsorted(r_group, r, side='right')

            if k < len(r_group):
                if np.random.rand() < P_two:
                    # Keep only one scene return behind spoof
                    remove_scene.extend(idx_group[k+1:].tolist())
                else:
                    # Spoof fully occludes behind
                    remove_scene.extend(idx_group[k:].tolist())
    print("unique remove_spoof:", len(set(remove_spoof)))
    print("unique remove_scene:", len(set(remove_scene)))

    # --- Apply removals ---
    mask_spoof = np.ones(spf_cur.shape[0], dtype=bool)
    mask_spoof[remove_spoof] = False
    spoof_points_rc = spf_cur[mask_spoof]
    
    removed_unique = len(set(remove_scene))
    kept_spoof = spoof_points_rc.shape[0]
    # print("expected points diff:", kept_spoof - removed_unique)

    mask_scene = np.ones(points.shape[0], dtype=bool)
    mask_scene[remove_scene] = False
    scene_points_rc = points[mask_scene]

    # Preserve lag column exactly as before
    lag_value = points[0, -1]
    lag_col = np.full((spoof_points_rc.shape[0], 1), lag_value, dtype=np.float32)    
    spf_pts_rc_lag = np.hstack([spoof_points_rc, lag_col])
    # helpers_ptt.plot_trace(spf_pts_rc_lag)

    frame_points_spoof_rc = np.concatenate([scene_points_rc, spf_pts_rc_lag], axis=0)

    return frame_points_spoof_rc




In [36]:
def process_4frames(dataset_mf, frame_idx, target_frames, spoof_points, frame_seq, perturbation, ptt_from_cp, dataset):
    #frame idx will be an index into centerpoint dataset
    # print(f"4frame point count before spoofing: {dataset_mf[frame_idx]['points'].shape}")
    points_frame = None
    #put global spoof in ego coordinates of lag = 0 frame
    none_spoofed = 1
    infos = dataset_mf.infos
    info_cur = infos[frame_idx]
    seq_name = info_cur['point_cloud']['lidar_sequence']
    pose_cur = info_cur['pose'].reshape(4, 4) #ego pose of current frame
    print(seq_name)
    
    
    #acquire list of frames in set of 4 (centerpoint multiframe) that need to be inspected for modification
    indices = []
    for i in range(4):
        target_idx = frame_idx - i

        if target_idx < 0:
            target_idx = 0

        if infos[target_idx]['point_cloud']['lidar_sequence'] != seq_name:
            target_idx = indices[-1] if indices else frame_idx

        indices.append(target_idx)
            
    print(f"cp 4 frames indices processed: {indices}")
    sum_indices = sum([target_frames[i] for i in indices])
    
    #if none of the frames are marked for spoofing, just return the original points for the current frame (no raycasting needed)
    if sum_indices == 0:
        print("not spoofing")
        points_frame = dataset[ptt_from_cp[frame_idx]]['points']
        return dataset_mf[frame_idx]['points'], none_spoofed, points_frame, pose_cur

    #iterate through 4 frames 
    merged_points_list = []
    for i, idx in enumerate(indices):
     
        # sample_idx = infos[idx]['point_cloud']['sample_idx']
        # points = isolate_frame_points(dataset_mf[idx]['points'], i)
        # points = isolate_frame_points(dataset_mf[frame_idx]['points'], i) # these are relative to the lag 0 frame
        
        # points = isolate_frame_points(dataset_mf[idx]['points'], 0).copy()
        points = dataset[ptt_from_cp[idx]]['points'].copy()
        points[:, -1] = (frame_idx - idx)*0.1
        
        pose_lag_i_frame = infos[idx]['pose'].reshape(4, 4)



        if(target_frames[idx]):
            #'points' pulled above is relative to lag 0 frame
            
            #1 convert to its own frame 
            # points_lag_i_frame_in_lag_0_coords = np.hstack((points[:, :3], np.ones(points.shape[0]).reshape(-1, 1)))                       
            # points_lag_i_frame_in_lag_i_coords = points_lag_i_frame_in_lag_0_coords @ (pose_cur).T @ np.linalg.inv(pose_lag_i_frame).T
            # points_lag_i_frame_in_lag_i_coords_full = np.hstack((points_lag_i_frame_in_lag_i_coords[:, :3], points[:, 3:]))  
            
            points_lag_i_frame_in_lag_i_coords_full = points    

            
            spoof_points_homo = None
            if(perturbation == "global_position_fixed"):
                
                #in global coordinates
                spoof_points_homo = np.hstack([spoof_points[:, 0:3], np.ones((spoof_points.shape[0], 1))]) 
                spf_cur = spoof_points_homo @ np.linalg.inv(pose_lag_i_frame).T
                
            elif(perturbation == "relative_position_fixed"):
                
                #in coordinates relative to ego at frame idx
                spoof_points_homo = np.hstack([spoof_points[:, 0:3], np.ones((spoof_points.shape[0], 1))])
                spf_cur = spoof_points_homo
            elif(perturbation == "random"):
                
                # in coordinates relative to ego at frame_idx
                spoof_points_homo = np.hstack([spoof_points[idx][:, 0:3], np.ones((spoof_points[idx].shape[0], 1))])
                spf_cur = spoof_points_homo
            
            # trace_pos_global_hom = np.hstack([trace_pos_global, np.ones((trace_pos_global.shape[0], 1))]) # put the global trace position into homogeneous coordinates
            
            
            src = spoof_points[idx] if isinstance(spoof_points, dict) else spoof_points
            spf_cur_lag_i = np.hstack([spf_cur[:, :3], src[:, 3:5]])
                                                                     
            #now need to do raycasting and append
            # at this point points_lag_i_frame_in_lag_i_coords_full are idx points in its own frame
            #               spf_cur_lag_i                           are spoof points relative to idx frame ego
            frame_points_spoof_rc = spoof_frame(spf_cur_lag_i, points_lag_i_frame_in_lag_i_coords_full)
            
            #use these for building the dataset
            # if idx == frame_idx, then this means this is lag 0, and it is already in lag 0 coordinates
            if(idx == frame_idx):
                points_frame = frame_points_spoof_rc
                                                           
            # convert these back to lag 0 coordintaes 
            frame_points_spoof_rc_hom = np.hstack((frame_points_spoof_rc[:, :3], np.ones(frame_points_spoof_rc.shape[0]).reshape(-1, 1)))
            frame_points_spoof_rc_lag0_coords = frame_points_spoof_rc_hom @ pose_lag_i_frame.T @ np.linalg.inv(pose_cur).T
                                                
            
            # print(f"spoofing frame {idx}, points diff = {frame_points_spoof_rc_lag0_coords.shape[0] - points.shape[0]}")
            points = np.hstack((frame_points_spoof_rc_lag0_coords[:, :-1], frame_points_spoof_rc[:, 3:]))
        else:
            # not spoofing, must convert back to lag 0 coordinates
            if(idx == frame_idx):
                points_frame = points
                
            points_last3cols = points[:, 3:]
            frame_points_hom = np.hstack((points[:, :3], np.ones(points.shape[0]).reshape(-1, 1)))
            frame_points_spoof_lag0_coords = frame_points_hom @ pose_lag_i_frame.T @ np.linalg.inv(pose_cur).T
            points = np.hstack((frame_points_spoof_lag0_coords[:, :-1], points_last3cols))
         
       
        # print(points.shape)
        merged_points_list.append(points)
    merged_points_array = np.vstack(merged_points_list)
    merged_points_array = merged_points_array.astype(np.float32)
    # print(f"4frame point count after spoofing: {merged_points_array.shape}")    

    
    none_spoofed = 0
    unique_lags = np.unique(np.round(merged_points_array[:, -1], 1))
    for lag in unique_lags:
        pts = merged_points_array[np.isclose(merged_points_array[:, -1], lag)]
        print(lag, pts.shape[0])
    if frame_seq == 31:
        lags_raw = merged_points_array[:, -1]
        print("lag raw min/max:", lags_raw.min(), lags_raw.max())
        u, c = np.unique(np.round(lags_raw, 3), return_counts=True)
        print("unique lags (rounded 1e-3):")
        for uu, cc in zip(u[:30], c[:30]):
            print(uu, cc)
        print("... total unique:", len(u))
        # helpers_ptt.plot_4frame_sequence(merged_points_array)
        # input("Press Enter to continue to the next frame...")
    return merged_points_array, none_spoofed, points_frame, pose_cur

In [37]:
def modify_frame_batch(dataset, 
                       dataset_mf, 
                       target_frames, 
                       frame, 
                       frame_seq, 
                       cp_mf_model, 
                       load_data_to_gpu, 
                       frame_ptt, 
                       ptt_from_cp,
                       spoof_points,
                       annotation,
                       perturbation):
    
    target_frames_keys = list(target_frames.keys())
    roi_boxes_dict = {}
    roi_scores_dict= {}
    roi_labels_dict = {}

    #get only 32 total frames. if frame_seq !=31 then there will be 35 total frames in there
    frames_iter_thru = target_frames_keys[3:] if frame_seq !=31 else target_frames_keys
    
    data_sample_poses = []
    print("target frames keys:")
    print(target_frames_keys)
    print("frames actually iterating thru:")
    print(frames_iter_thru)
    
    #iterate through 32 frames and apply spoofing to frames marked for spoofing in target_frames. 
    # also get ROIs from centerpoint multiframe model for all 32 frames (these will be used as input to ptt model)
    points_dict = {}
    gt_boxes_dict = {}
    for frame_idx in frames_iter_thru:
        
        #inspect/perturb past 4 frames in centerpoint multiframe 
        if(type(spoof_points) != dict):
            print(f"num spoof points: {spoof_points.shape[0]}")
        elif(target_frames[frame_idx]):
            print(f"num spoof points : {spoof_points[frame_idx].shape[0]}")
            
        frame_points, none_spoofed, frame_idx_points, frame_idx_pose = process_4frames(dataset_mf, 
                                                                                       frame_idx, 
                                                                                       target_frames, 
                                                                                       spoof_points, 
                                                                                       frame_seq, 
                                                                                       perturbation,
                                                                                       ptt_from_cp,
                                                                                       dataset)
        points_dict[frame_seq - 31 + (frame_idx - frames_iter_thru[0])] = frame_idx_points
        data_sample_poses.append(frame_idx_pose)
        gt_boxes_dict[frame_seq - 31 + (frame_idx - frames_iter_thru[0])] = dataset[ptt_from_cp[frame_idx]]['gt_boxes']
        
        #!select current frame's dictionary for modification 
        dict_mf_mod = dataset_mf[frame_idx]
        dict_mf_mod['batch_size']   = 1

        #! only run voxelization and centerpoint model if there is spoofing in any of the 4 frames being processed. 
        #! if not, just use the original roi boxes, scores, and labels from the dataset for that frame 
        if(none_spoofed == 0):
            print("================================MAKING CPMF PREDICTION===============================")

            #!replace points in the dict with the new spoofed points (after raycasting) for the past 4 frames
            dict_mf_mod['points'] = frame_points
            #! rerun v oxelization and forward pass

            for k in ['voxels', 'voxel_coords', 'voxel_num_points']:
                dict_mf_mod.pop(k, None)
            dict_mf_mod   = helpers_ptt.inject_gt_names(dict_mf_mod, dataset_mf.class_names)
            dict_mf_mod   = dataset_mf.prepare_data(dict_mf_mod)

            load_data_to_gpu(dict_mf_mod)
            batch_mf_mod = helpers_ptt.convert_to_batch_cp_mf(dict_mf_mod)

            cp_mf_model.eval()
            with torch.no_grad():
                pred_mod_mf, _   = cp_mf_model(batch_mf_mod)
            roi_boxes_dict[frame_idx] = pred_mod_mf[0]['pred_boxes']
            roi_scores_dict[frame_idx] = pred_mod_mf[0]['pred_scores']
            roi_labels_dict[frame_idx] = pred_mod_mf[0]['pred_labels']

        else:

            #replace with batch frame_idx logic for final (sequences need to align)
            #frame - 31 = first index in sequence --> corresponds to last element in boxes array
            #frame_idx - (frame-31)
            
            #! otherwise just use the original ptt data
            #ptt pred boxes is in order of current frame, frame-1, frame-2, ..., frame-31. 
            # need to index into this array to get the right pred boxes for the right frame.
            
            which_pred = frame-frame_idx
            # example:
            # frame = 51, frame_idx = 51--> which_pred = 0 (current frame)
            # frame = 51, frame_idx = 20--> which_pred = 31 (last element in array, corresponds to frame-31)
            
            
            frame_idx_ptt = ptt_from_cp[frame_idx]
            
            #!reminder: after a certain point, the frame indices for ptt and centerpoint multiframe don't refer to the same segment
            #! so you have to use the mapping from matches to get the corresponding frame index in ptt for the given centerpoint multiframe frame index.
            #! this ensures that both indices are referring to the same frame within a trajectory (segment)
            frame_data = dataset[frame_idx_ptt]
            print(f"frame_idx: {frame_idx}, which_pred: {which_pred}", frame_data['roi_boxes'][which_pred].shape)
            roi_boxes_dict[frame_idx] = torch.from_numpy(
                                            frame_data['roi_boxes'][which_pred]
                                        ).to('cuda:0')

            roi_scores_dict[frame_idx] = torch.from_numpy(
                                            frame_data['roi_scores'][which_pred]
                                        ).to('cuda:0')

            roi_labels_dict[frame_idx] = torch.from_numpy(
                                            frame_data['roi_labels'][which_pred]
                                        ).to('cuda:0')
    
    annotation['frame_points'] = points_dict
    annotation['frame_poses'] = np.array(data_sample_poses)
    annotation['frame_gt_boxes'] = gt_boxes_dict
    
    
    max_preds = max(tensor.size(0) for tensor in roi_boxes_dict.values())
    # max_preds = 65
    # print(max_preds)
    past_boxes = []
    past_scores = []
    past_labels = []

    #the roi boxes structure has roi_cur_frame, roi_cur_frame-1, etc

    just_frames = frames_iter_thru[::-1]
    # print(just_frames)
    # just_frames = [frame_idx for frame_idx in target_frames]
    # just_frames.reverse()

    #now iterate through target frames in reverse order: frame, frame-1... frame-31 
    # pad the roi boxes, scores, and labels for each frame to have the same number of predictions (max_preds) and then stack them into a single tensor for input into ptt model
    for frame_idx in just_frames:
        preds = roi_boxes_dict[frame_idx]
        scores = roi_scores_dict[frame_idx]
        labels = roi_labels_dict[frame_idx]
        num_preds = preds.size(0)
        pad_size = max_preds - num_preds
        padded_preds = F.pad(preds, (0, 0, 0, pad_size), mode = 'constant', value = 0)
        padded_scores = F.pad(scores, (0, pad_size), mode = 'constant', value = 0)
        padded_labels = F.pad(labels, (0, pad_size), mode = 'constant', value = 0)
        past_boxes.append(padded_preds)
        past_scores.append(padded_scores)
        past_labels.append(padded_labels)
    past_boxes = torch.stack(past_boxes,dim = 0).detach().cpu().numpy()
    past_scores = torch.stack(past_scores, dim = 0).detach().cpu().numpy()
    past_labels = torch.stack(past_labels, dim = 0).detach().cpu().numpy()
    
    #! now modify the ptt batch for the current frame with the new roi boxes, scores, and labels obtained from the centerpoint multiframe model for the past 32 frames
    dict_mod_ptt = dataset[frame_ptt]
    dict_mod_ptt['roi_boxes'] = past_boxes
    dict_mod_ptt['roi_scores'] = past_scores
    dict_mod_ptt['roi_labels'] = past_labels

    # add spoofed points to scene for single frame ptt. they might be used for something or the other
    if target_frames[frame]:
        infos = dataset.infos
        info_cur = infos[frame_ptt]
        pose_cur = info_cur['pose'].reshape(4, 4) #ego pose of current frame
        
        if(perturbation == "global_position_fixed"):
                
            #in global coordinates
            spoof_points_homo = np.hstack([spoof_points[:, 0:3], np.ones((spoof_points.shape[0], 1))]) 
            spf_cur = spoof_points_homo @ np.linalg.inv(pose_cur).T

        elif(perturbation == "relative_position_fixed"):

            #in coordinates relative to ego at frame idx
            spoof_points_homo = np.hstack([spoof_points[:, 0:3], np.ones((spoof_points.shape[0], 1))])
            spf_cur = spoof_points_homo
        elif(perturbation == "random"):

            # in coordinates relative to ego at frame_idx
            spoof_points_homo = np.hstack([spoof_points[frame_seq][:, 0:3], np.ones((spoof_points[frame_seq].shape[0], 1))])
            spf_cur = spoof_points_homo
            
        src = spoof_points[frame_seq] if isinstance(spoof_points, dict) else spoof_points
        spf_cur_nolag = np.hstack([spf_cur[:, :3], src[:, 3:5]])

        points = dataset[frame_ptt]['points']

        # frame_points_spoof_rc = spoof_frame(spf_cur_nolag, points[:, :-1])
        # zeros = np.zeros(frame_points_spoof_rc.shape[0])
        # dict_mod_ptt['points'] = np.column_stack((frame_points_spoof_rc, zeros))

        frame_points_spoof_rc = spoof_frame(spf_cur_nolag, points)
        dict_mod_ptt['points'] = frame_points_spoof_rc



    # print(dict_mod_ptt['points'].shape)
    dict_mod_ptt = helpers_ptt.inject_gt_names(dict_mod_ptt, dataset.class_names)
    dict_mod_ptt = dataset.prepare_data(dict_mod_ptt)
    batch_mod_ptt = dataset.collate_batch([dict_mod_ptt])

    return batch_mod_ptt, annotation 

spoofing
- only valid frames to spoof are frames 31 - last frame in a sequence
- do not do spoofing for frames 0-30 in a sequence
- every five, for example (suppose last one is 197): 31, 36, 41, ... 191, 196

In [38]:
# traces_file = 'traces.pkl'
# # Open the file in read-binary mode ('rb')
# with open(traces_file, 'rb') as file:
#     # Use pickle.load() to deserialize the data
#     traces = pkl.load(file)
# N_POINTS_TRACE = 100
# trace = next((d for d in traces if d.get('num_points') == N_POINTS_TRACE), None)
# centered_trace = center_trace(trace)
# ego_centric_pos = np.array([15, 0])
# front_near_trace_planar = centered_trace + ego_centric_pos
# front_near_trace = np.concatenate([front_near_trace_planar, trace['points'][:, 2].reshape(-1, 1)], axis = 1)
# ego_centric_trace_hom = np.vstack([ego_centric_pos.reshape(-1, 1), trace['box'][2], 1]) # trace centroid in global homogeneous coordinates


In [39]:
# helpers_ptt.plot_trace(trace['points'])

In [40]:
# # start_time = time.perf_counter()

# START_SPOOF_FROM_IDX = 16 #out of frames 0-31 in a sequenxe of 32, starting from which index do you want to spoof?
# target_frame = {}
# frame = 41
# for i in range(frame-34, frame+1):
#     if(i <=frame - START_SPOOF_FROM_IDX):
#         if(i <0): target_frame[0] = False
#         else: target_frame[i] = False
#     else: target_frame[i] = True
# # print(f"The code took {execution_time} seconds to execute.")

In [41]:
# cp_map = {
# info['frame_id']: i
# for i, info in enumerate(dataset_mf.infos)
# }
# matches = [
#     (i, cp_map[info['frame_id']])
#     for i, info in enumerate(dataset.infos)
#     if info['frame_id'] in cp_map
# ]

# ptt_from_cp = {
#     cp_idx: ptt_idx
#     for ptt_idx, cp_idx in matches
# }

In [42]:
# import numpy as np
import random

def build_target_frames(
    frame_seq: int,
    frame: int,
    mode: str,
    min_spoof: int,
    max_spoof: int,
    dataset_mf : dict, 
    traces : dict,
    segment_name: str,
    dataset_name: str,
    perturbation: str
):
    """create a dictionary that tells which frames (in cp mf indices) to spoof.
       additionally create an annotations dict for the 32 frame datasample

    Args:
        selected_trace (): 

    Returns:
        _type_: _description_
    """
    # print(mode == "last_only")
    #must also return an annotation of the data sample
    
    #which segment
    #which window of frames within segment 
    #which frames within window are being spoofed
    # dataset: clean history, perturbed observation
    #          perturbed history, clean observation
    #          perturbed history, perturbed observation
    
    #perturbation: global position fixed
    #              relative position fixed
    #              random 
    
    #potentially later: spoofed point data
    #npoints : (some number)
    #           random if on random setting 
    #position: global coordinate if global spoofing
    #          relative coordinate if relative spoofing 
    
    annotation = {
        'segment': segment_name,
        'dataset': dataset_name,
        'perturbation': perturbation,
        'frame_window': [frame_seq-31, frame_seq],   
    }
    
    rng = np.random.default_rng()
    
    #for global position fixed, the spoofed object will be this many meters ahead of the final frame (frame_seq), but converted to global coords
    # for relative position spoof, the spoofed object will remain this distance ahead of ego for each frame marked for spoofing 
    trace = rng.choice(traces)
    relative_position_offset= np.array([rng.uniform(8, 10), rng.uniform(-1, 1) ]) 
    
    print(f"placing in relative position : {relative_position_offset}")
    
    #bring to ego-frame origin
    centered_trace = center_trace(trace)
    
    #shift it to randomly generated position
    front_near_trace_planar = centered_trace + relative_position_offset
    
    #add the z axis 
    front_near_trace = np.concatenate([front_near_trace_planar, trace['points'][:, 2].reshape(-1, 1)], axis = 1)
    
    
    #add reflection and elongations 
    
    spoof_points = None
    
    #pose of last (current) frame in 32 frame sequence
    # used for global placement if spoofing last frame
    anchor_pose = dataset_mf.infos[frame]['pose']


    
    target_frames = {}

    # -------------------------
    # Case 1: frame_seq == 31
    # -------------------------
    if frame_seq == 31:

        window_size = 32
        offsets = np.arange(window_size)

        # ----- Mode 5: last_only -----
        if mode == "last_only":
            for off in offsets:
                cp_idx = frame - 31 + off
                target_frames[cp_idx] = (off == 31)
            # return target_frames

        # ----- Determine number of spoofed frames -----
        elif mode.startswith("contig"):
            L = rng.integers(min_spoof, max_spoof + 1)

            if mode == "contig_to_last":
                end_off = 31
                start_off = end_off - (L - 1)

            elif mode == "contig_random_exclude_last":
                max_off = 30  # exclude last
                start_off = rng.integers(0, max_off - L + 2)
                end_off = start_off + L - 1

            else:
                raise ValueError("Invalid contiguous mode")

            for off in offsets:
                cp_idx = frame - 31 + off
                target_frames[cp_idx] = (start_off <= off <= end_off)
                if(mode == "contig_random_exclude_last" and off == end_off):
                    
                    #if doing "perturbed history, clean observation, anchor spoof relative to last frame marked for spoofing
                    anchor_pose = dataset_mf.infos[cp_idx]['pose']
                    # print(f"anchoring relative to cp frame{cp_idx}")


        elif mode.startswith("random"):

            L = rng.integers(min_spoof, max_spoof + 1)

            if mode == "random_include_last":
                spoof_set = {31}
                remaining = L - 1
                candidates = np.arange(0, 31)
                if remaining > 0:
                    chosen = rng.choice(candidates, size=remaining, replace=False)
                    spoof_set.update(chosen.tolist())


            elif mode == "random_exclude_last":
                candidates = np.arange(0, 31)
                spoof_set = set(
                    rng.choice(candidates, size=L, replace=False).tolist()
                )


            else:
                raise ValueError("Invalid random mode")

            for off in offsets:
                cp_idx = frame - 31 + off
                target_frames[cp_idx] = (off in spoof_set)
                
                
            #build dict of spoofs: 

        else:
            raise ValueError("Invalid mode")
        
        
        
        target_ptt_segment_indices = sorted([
            off for off in range(32)
            if target_frames[frame - 31 + off]
        ])
        # print(target_ptt_segment_indices)
        # annotation['target_frames'] = [item + (frame_seq - 31) for item in target_ptt_segment_indices]
        annotation['target_frames_win'] = target_ptt_segment_indices
        annotation['target_frames_seg'] = [i + (frame_seq-31) for i in target_ptt_segment_indices]

        
        
        if(perturbation == 'global_position_fixed'):
            trace_pos_global = (front_near_trace @ anchor_pose[:3, :3].T) + anchor_pose[:, -1][:3]
            spoof_points = np.hstack((trace_pos_global, trace['points'][:, 3:5]))
            annotation['spoof_points'] = spoof_points
        
        elif(perturbation == 'relative_position_fixed'):
            spoof_points = np.hstack((front_near_trace, trace['points'][:, 3:5]))
            annotation['spoof_points'] = spoof_points
        
        elif(perturbation == 'random'):
            # form a dictionary of 
            spoof_points = {}
            
            anno_points = {}
          
            # change to segment index target frames 
            target_frames_true = [i for i in target_frames.keys() if target_frames[i]]
            for (i,j) in zip(target_frames_true, annotation['target_frames_seg']):
                # print(i, j)
                # print(target_frames[i])

                trace = rng.choice(traces)

                relative_position_offset= np.array([rng.uniform(8, 20), rng.uniform(-3, 3) ]) 
                centered_trace = center_trace(trace)
                front_near_trace_planar = centered_trace + relative_position_offset
                front_near_trace = np.hstack((front_near_trace_planar, trace['points'][:, 2:5]))
                
                spoof_points[i] = front_near_trace
                anno_points[j] = front_near_trace
            annotation['spoof_points']= anno_points
        annotation['perturbation'] = perturbation

        #also return set of spoofed points. if random, it will be a dict 
        return target_frames, annotation, spoof_points, perturbation

    # -------------------------
    # Case 2: frame_seq > 31
    # -------------------------
    else:

        # Full CP window: 35 frames
        # j = 0,1,2 always False
        for j in range(3):
            cp_idx = frame - 34 + j
            target_frames[cp_idx] = False

        ptt_indices = np.arange(3, 35)  # 32 frames

        # ----- Mode 5: last_only -----
        if mode == "last_only":
            for j in ptt_indices:
                cp_idx = frame - 34 + j
                target_frames[cp_idx] = (j == 34)
            # return target_frames

        # ----- Determine number of spoofed frames -----
        elif mode.startswith("contig"):
            L = rng.integers(min_spoof, max_spoof + 1)

            if mode == "contig_to_last":
                end_j = 34
                start_j = end_j - (L - 1)

            elif mode == "contig_random_exclude_last":
                max_j = 33  # exclude last
                start_j = rng.integers(3, max_j - L + 2)
                end_j = start_j + L - 1

            else:
                raise ValueError("Invalid contiguous mode")

            for j in ptt_indices:
                cp_idx = frame - 34 + j
                target_frames[cp_idx] = (start_j <= j <= end_j)
                if(mode == "contig_random_exclude_last" and j == end_j):
                    anchor_pose = dataset_mf.infos[cp_idx]['pose']
                    # print(f"anchoring relative to cp frame{cp_idx}")

        elif mode.startswith("random"):

            L = rng.integers(min_spoof, max_spoof + 1)

            if mode == "random_include_last":
                spoof_set = {34}
                remaining = L - 1
                candidates = np.arange(3, 34)
                if remaining > 0:
                    chosen = rng.choice(candidates, size=remaining, replace=False)
                    spoof_set.update(chosen.tolist())

            elif mode == "random_exclude_last":
                candidates = np.arange(3, 34)
                spoof_set = set(
                    rng.choice(candidates, size=L, replace=False).tolist()
                )
                

            else:
                raise ValueError("Invalid random mode")
            for j in ptt_indices:
                cp_idx = frame - 34 + j
                target_frames[cp_idx] = (j in spoof_set)

        else:
            raise ValueError("Invalid mode")
            

            
        
        target_ptt_segment_indices = sorted([
            j - 3 for j in range(3, 35)
            if target_frames[frame - 34 + j]
        ])
        # print(target_ptt_segment_indices)

        # annotation['target_frames'] = [item + (frame_seq - 31) for item in target_ptt_segment_indices]
        annotation['target_frames_win'] = target_ptt_segment_indices
        annotation['target_frames_seg'] = [i + (frame_seq-31) for i in target_ptt_segment_indices]
        
        #also return set of spoofed points. if random, it will be a dict 
        if(perturbation == 'global_position_fixed'):
            trace_pos_global = (front_near_trace @ anchor_pose[:3, :3].T) + anchor_pose[:, -1][:3]
            spoof_points = np.hstack((trace_pos_global, trace['points'][:, 3:5]))
            annotation['spoof_points'] = spoof_points
        
        elif(perturbation == 'relative_position_fixed'):
            spoof_points = np.hstack((front_near_trace, trace['points'][:, 3:5]))
            annotation['spoof_points'] = spoof_points

        elif(perturbation == 'random'):
            # form a dictionary of 
            spoof_points = {}
            
            anno_points = {}
          
            # change to segment index target frames 
            target_frames_true = [i for i in target_frames.keys() if target_frames[i]]
            for (i,j) in zip(target_frames_true, annotation['target_frames_seg']):
                # print(i, j)
                # print(target_frames[i])

                trace = rng.choice(traces)

                relative_position_offset= np.array([rng.uniform(8, 20), rng.uniform(-5, 5) ]) 
                centered_trace = center_trace(trace)
                front_near_trace_planar = centered_trace + relative_position_offset
                front_near_trace = np.hstack((front_near_trace_planar, trace['points'][:, 2:5]))

                spoof_points[i] = front_near_trace
                anno_points[j] = front_near_trace
            annotation['spoof_points']= anno_points
        annotation['perturbation'] = perturbation
        # print(annotation['target_frames'])
        #also return set of spoofed points. if random, it will be a dict 
        return target_frames, annotation, spoof_points, perturbation



In [43]:
traces_file = 'traces.pkl'
# Open the file in read-binary mode ('rb')
with open(traces_file, 'rb') as file:
    # Use pickle.load() to deserialize the data
    traces = pkl.load(file)

target_frames , annotation, spoof_points, perturbation= build_target_frames(
    frame_seq=196,
    frame=5000,
    mode= 'contig_random_exclude_last',
    min_spoof=4,
    max_spoof=16,
    dataset_mf = dataset_mf,
    traces = traces,
    segment_name = 'waymo_segment_0014',
    dataset_name = 'perturbed_history_perturbed_observation',
    perturbation = 'relative_position_fixed'
)

#perturbations: 'global_position_fixed', 'relative_position_fixed', 'random'
#mode : contig_to_last, contig_random_exclude_last, random_include_last, random_exclude_last, last_only

# print(target_frames)

placing in relative position : [8.78027506 0.28348719]


In [44]:
annotation.keys()

dict_keys(['segment', 'dataset', 'perturbation', 'frame_window', 'target_frames_win', 'target_frames_seg', 'spoof_points'])

In [45]:
helpers_ptt.avg_spoof_range(spoof_points)

9.479502675790533

In [46]:
np.mean(spoof_points[:, 3])

0.11752852985421114

In [49]:
# annotation

In [50]:
cfg_ptt = cfg

In [51]:
# annotation

In [52]:
# annotation 

In [53]:
  # frame_seq: int,
  #   frame: int,
  #   mode: str,
  #   min_spoof: int,
  #   max_spoof: int,
  #   dataset_mf : dict, 
  #   traces : dict,
  #   segment_name: str,
  #   dataset_name: str,
  #   perturbation: str

In [54]:
importlib.reload(helpers_ptt)

<module 'helpers_ptt' from '/storage/scratch1/9/spanse30/PTT/helpers_ptt.py'>

In [220]:
traces[0].keys()

dict_keys(['frame', 'box', 'points', 'num_points'])

In [221]:
import time

In [233]:


#!important
#indices into the cp dataset and ptt dataset start referring to different frames after a certain point
# create a mapping between cp and ptt indices such that both are referring to the same frame within the segment
cp_map = {
info['frame_id']: i
for i, info in enumerate(dataset_mf.infos)
}
matches = [
    (i, cp_map[info['frame_id']])
    for i, info in enumerate(dataset.infos)
    if info['frame_id'] in cp_map
]

ptt_from_cp = {
    cp_idx: ptt_idx
    for ptt_idx, cp_idx in matches
}

STRIDE = 16 # do spoofing on every xth frame in ptt waymo

traces_file = 'traces.pkl'
# Open the file in read-binary mode ('rb')
with open(traces_file, 'rb') as file:
    # Use pickle.load() to deserialize the data
    traces = pkl.load(file)
traces = [t for t in traces if t['num_points'] >= 100]


frame_seq = 0

#bounding box predictions for the segment
segment_preds = [] 

#spoofed data samples per segment 
# with stride = 5, should be ~6 samples per segment 
segment_dataset = []

current_segment = None

# save_dir_preds = args.pred_save_dir
# os.makedirs(save_dir_preds, exist_ok=True)
# save_dir_dataset = args.dataset_save_dir
# os.makedirs(save_dir_dataset, exist_ok=True)
save_dir_preds = 'experiments/e1/preds'
os.makedirs(save_dir_preds, exist_ok=True)
save_dir_dataset = 'experiments/e1/dataset'
os.makedirs(save_dir_dataset, exist_ok=True)

# save_dir_preds = 'segment_checkpoints_e2'
# save_dir_dataset = None

# elevations = np.load('waymo_top_lidar_inclinations.npy')
# stats = np.load("range_conditioned_stats.npy", allow_pickle=True).item()

# if predictions have already been written for a segment, dont overwrite and go to next segment
completed = set(
    f[:-4] for f in os.listdir(save_dir_preds)
)

#iterate thru every frame in val split
try: 
    for (i_ptt,batch)  in enumerate(test_loader):
        if(i_ptt == 197):
            print("reached end of segment 1")
            break
        #get the frame index in centerpoint indices
        frame_ptt, frame = matches[i_ptt] 
        seq_ptt = batch['frame_id'][0]

        #frames must be the same... ptt indices and centerpoint mf indices diverge after a certain point
        assert seq_ptt == dataset_mf.infos[frame]['frame_id']

        #get just the segment name
        segment = seq_ptt.rsplit("_", 1)[0]
        print(segment)
        
        #skip if data for this segment has already been written to disk
        if segment in completed:
            continue
                
        #only happens on the very first iteration 
        if current_segment is None:
            current_segment = segment
            
        #once you have crossed into a new segment, write previous segment's data to disk
        if segment != current_segment:
            
            pred_path = f"{save_dir_preds}/{current_segment}.pkl"
            dataset_path = f"{save_dir_dataset}/{current_segment}.pkl"
            with open(pred_path,"wb") as f:
                pkl.dump(segment_preds,f)
            with open(dataset_path,"wb") as f:
                pkl.dump(segment_dataset,f)
            
            #log that data was saved
            # log.info("Saved: %s", current_segment)
            # print(f"saved {current_segment}")
            
            # emtpy out dictionaries for next segment
            segment_preds = []
            segment_dataset = []
            
            #update segment 
            current_segment = segment

        # pull the frame within the segment from segment name
        frame_seq = int(seq_ptt.rsplit("_", 1)[-1])

        # spoof every xth frame (set by stride), starting only from frame index 31 (frame 32)
        if(frame_seq > 30 and not(frame_seq - 31)%STRIDE): 
            
            #1
            #target_frames: dict of frames (centerpoint indices) with T/F if spoofing
            # annotation: segment name, dataset name, perturbation type, window being spoofed (relative to segment), frames within window (relative to 32 frame datasample)
            #             points (dict with target_frame (relative to 32 frame sequence : spoofed points      
            target_frames , annotation, spoof_points, perturbation= build_target_frames(
                                                                        frame_seq=frame_seq,
                                                                        frame=frame,
                                                                        mode='contig_random_exclude_last',
                                                                        min_spoof=4,
                                                                        max_spoof=16,
                                                                        dataset_mf = dataset_mf,
                                                                        traces = traces,
                                                                        segment_name = current_segment,
                                                                        dataset_name = 'perturbed_history_perurbed_observation',
                                                                        perturbation = 'global_position_fixed'
                                                                    )
            # print(f"frame_seq: {frame_seq} | frame = {frame} | frame_ptt: {frame_ptt}")
            # print(target_frames)
            
            #3 get spoofed_batch
            batch, annotation = modify_frame_batch(dataset, 
                                       dataset_mf, 
                                       target_frames, 
                                       frame, 
                                       frame_seq, 
                                       cp_mf_model, 
                                       load_data_to_gpu, 
                                       frame_ptt, 
                                       ptt_from_cp, 
                                       spoof_points,
                                       annotation,
                                       perturbation)
            print(annotation.keys())
            break
            # print(annotation.keys())
            ##############NEED TO WRITE ANNOTATION TO DISK####################
            
            # print(frame_seq, target_frames)
            # print(seq_ptt, frame_seq, frame, min(target_frames), max(target_frames), max(target_frames)-min(target_frames))

            # log.info(f"spoofed frame {frame_seq} of segment {current_segment}")
            segment_dataset.append(annotation)
            
            
        load_data_to_gpu(batch)

        with torch.no_grad():
            pred_dicts, _ = model(batch)

        print("================================MAKING PTT PREDICTION===============================")
        annos = dataset.generate_prediction_dicts(
            batch,
            pred_dicts,
            cfg_ptt.CLASS_NAMES
        )

        segment_preds+=annos
        
except Exception:
    # log.error("===== EXCEPTION =====")
    # log.error(traceback.format_exc())
    raise  

# last segment
# if segment_preds:
#     pred_path = f"{save_dir_preds}/{current_segment}.pkl"
#     dataset_path = f"{save_dir_dataset}/{current_segment}.pkl"
#     with open(pred_path,"wb") as f:
#         pkl.dump(segment_preds,f)
#     with open(dataset_path,"wb") as f:
#         pkl.dump(segment_dataset,f)



segment-10203656353524179475_7625_000_7645_000_with_camera_labels
================================MAKING PTT PREDICTION===============================
segment-10203656353524179475_7625_000_7645_000_with_camera_labels
================================MAKING PTT PREDICTION===============================
segment-10203656353524179475_7625_000_7645_000_with_camera_labels
================================MAKING PTT PREDICTION===============================
segment-10203656353524179475_7625_000_7645_000_with_camera_labels
================================MAKING PTT PREDICTION===============================
segment-10203656353524179475_7625_000_7645_000_with_camera_labels
================================MAKING PTT PREDICTION===============================
segment-10203656353524179475_7625_000_7645_000_with_camera_labels
================================MAKING PTT PREDICTION===============================
segment-10203656353524179475_7625_000_7645_000_with_camera_labels
============================

In [ ]:
spoof_points

In [ ]:
helpers_ptt.plot_4frame_sequence(dataset_mf[1022]['points'])

In [ ]:

file_path = 'segment_checkpoints_e1/segment-10289507859301986274_4200_000_4220_000_with_camera_labels.pkl' # Replace with the actual path to your PKL file

try:
    # Open the file in read-binary mode
    with open(file_path, 'rb') as file:
        data = pkl.load(file)
    
    # Now 'data' contains the deserialized Python object
    print("Data loaded successfully:")

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
print(dataset[ptt_from_cp[0]]['points'].shape[0])

In [ ]:
# completed = set(
#     f[:-4] for f in os.listdir(save_dir)
# )

In [ ]:
# completed

In [ ]:
# seq_ptt

In [ ]:
# batch.keys()

In [ ]:
# batch['frame_id']

In [ ]:
# dataset[0].keys()

In [ ]:
# frame = 50
# centered_trace = center_trace(trace)
# ego_centric_pos = np.array([15, 0])
# front_near_trace_planar = centered_trace + ego_centric_pos
# front_near_trace = np.concatenate([front_near_trace_planar, trace['points'][:, 2].reshape(-1, 1)], axis = 1)
# ego_centric_trace_hom = np.vstack([ego_centric_pos.reshape(-1, 1), trace['box'][2], 1]) # trace centroid in global homogeneous coordinates

# current_pose = dataset_mf.infos[frame]['pose']
# global_trace_centroid_hom = (current_pose @ ego_centric_trace_hom).reshape(-1)

# target_frames = {}
# for i in range(frame-34, frame+1):
#     if(i <=frame - START_SPOOF_FROM_IDX):
#         if(i <0): target_frames[0] = False
#         else: target_frames[i] = False
#     else: target_frames[i] = True

# poses = []
# for i in range(frame - 34, frame+1):
#     pose = dataset_mf.infos[i]['pose'][:, -1]
#     poses.append(pose)
# poses_array = np.stack(poses)
# threshold = 47#change this 
# target_frames = {}

# #automatically mark first 3 as false
# for i, pose in enumerate(poses_array):
#     frame_ = i + (frame - 34) if frame-34>=0 else 0
#     if(frame_ <  frame-31 or np.linalg.norm(global_trace_centroid_hom - pose) > threshold):
#         target_frames[frame_] = False
#     else:
#         target_frames[frame_] = True

In [ ]:
# target_frames

In [ ]:
# start_time = time.perf_counter()

# centered_trace = center_trace(trace)
# ego_centric_pos = np.array([15, 0])
# front_near_trace_planar = centered_trace + ego_centric_pos
# front_near_trace = np.concatenate([front_near_trace_planar, trace['points'][:, 2].reshape(-1, 1)], axis = 1)
# ego_centric_trace_hom = np.vstack([ego_centric_pos.reshape(-1, 1), trace['box'][2], 1]) # trace centroid in global homogeneous coordinates

# current_pose = dataset_mf.infos[frame]['pose']
# global_trace_centroid_hom = (current_pose @ ego_centric_trace_hom).reshape(-1)

In [ ]:
# #put trace in global coordinates
# trace_pos_global = (front_near_trace @ current_pose[:3, :3].T) + current_pose[:, -1][:3]#front_near_trace + global_trace_centroid_hom[:3]

# target_frames_keys = list(target_frames.keys())
# roi_boxes_dict = {}
# roi_scores_dict= {}
# roi_labels_dict = {}

# CP_BATCH = 8

# cp_inputs = []
# cp_indices = []

# #iterate through past 32 frames
# frames_iter_thru = target_frames_keys[3:] if frame !=31 else target_frames_keys
# print(frames_iter_thru)
# # print(frames_iter_thru)
# for frame_idx in frames_iter_thru:
#     # print(frame_idx)
#     frame_points, n_spoof_rc, none_spoofed= process_4frames(dataset_mf, frame_idx, target_frames, trace_pos_global)
    
#     dict_mf_mod = dataset_mf[frame_idx]
#     dict_mf_mod['batch_size']   = 1

#     if(none_spoofed == 0):
#         # print(f"frame_idx = {frame_idx}, none_spoofed = {none_spoofed}")
#         dict_mf_mod['points'] = frame_points

#         for k in ['voxels', 'voxel_coords', 'voxel_num_points']:
#             dict_mf_mod.pop(k, None)
#         dict_mf_mod   = helpers_ptt.inject_gt_names(dict_mf_mod, dataset_mf.class_names)
#         dict_mf_mod   = dataset_mf.prepare_data(dict_mf_mod)

#         load_data_to_gpu(dict_mf_mod)
#         batch_mf_mod = helpers_ptt.convert_to_batch_cp_mf(dict_mf_mod)

#         cp_mf_model.eval()
#         with torch.no_grad():
#             pred_mod_mf, _   = cp_mf_model.forward(batch_mf_mod)
#         roi_boxes_dict[frame_idx] = pred_mod_mf[0]['pred_boxes']
#         roi_scores_dict[frame_idx] = pred_mod_mf[0]['pred_scores']
#         roi_labels_dict[frame_idx] = pred_mod_mf[0]['pred_labels']
        
# #         dict_mf_mod = dataset_mf[frame_idx]
# #         dict_mf_mod['points'] = frame_points

# #         for k in ['voxels','voxel_coords','voxel_num_points']:
# #             dict_mf_mod.pop(k, None)

# #         dict_mf_mod = helpers_ptt.inject_gt_names(
# #             dict_mf_mod, dataset_mf.class_names
# #         )

# #         dict_mf_mod = dataset_mf.prepare_data(dict_mf_mod)

# #         cp_inputs.append(dict_mf_mod)
# #         cp_indices.append(frame_idx)

# #         # ---------- RUN BATCH ----------
# #         if len(cp_inputs) == CP_BATCH:
# #             print(cp_indices)

# #             batch = dataset_mf.collate_batch(cp_inputs)
# #             load_data_to_gpu(batch)

# #             with torch.no_grad():
# #                 preds, _ = cp_mf_model(batch)

# #             for i, fidx in enumerate(cp_indices):
# #                 roi_boxes_dict[fidx]  = preds[i]['pred_boxes']
# #                 roi_scores_dict[fidx] = preds[i]['pred_scores']
# #                 roi_labels_dict[fidx] = preds[i]['pred_labels']

# #             cp_inputs.clear()
# #             cp_indices.clear()
        
#         # print(f"frame_idx: {frame_idx}, shape: {pred_mod_mf[0]['pred_boxes'].device}")

#     else:
#         # print(f"frame_idx = {frame_idx}, none_spoofed = {none_spoofed}")

#         #replace with batch frame_idx logic for final (sequences need to align)
#         #frame - 31 = first index in sequence --> corresponds to last element in boxes array
#         #frame_idx - (frame-31)
#         which_pred = 31 - (frame_idx - (frame-31))
#         # print(f"frame_idx: {frame_idx}, frame_idx - frame + 31: {(frame_idx - (frame-31))}, which_pred: {which_pred}")
#         frame_data = dataset[frame_idx]
#         # print(f"frame_idx: {frame_idx}, which_pred: {which_pred}", frame_data['roi_boxes'][which_pred].shape)
#         roi_boxes_dict[frame_idx] = torch.from_numpy(
#                                         frame_data['roi_boxes'][which_pred]
#                                     ).to('cuda:0')

#         roi_scores_dict[frame_idx] = torch.from_numpy(
#                                         frame_data['roi_scores'][which_pred]
#                                     ).to('cuda:0')

#         roi_labels_dict[frame_idx] = torch.from_numpy(
#                                         frame_data['roi_labels'][which_pred]
#                                     ).to('cuda:0')
# # if cp_inputs:
# #     print(cp_indices)
# #     batch = dataset_mf.collate_batch(cp_inputs)
# #     load_data_to_gpu(batch)

# #     with torch.no_grad():
# #         preds, _ = cp_mf_model(batch)

# #     for i, fidx in enumerate(cp_indices):
# #         roi_boxes_dict[fidx]  = preds[i]['pred_boxes']
# #         roi_scores_dict[fidx] = preds[i]['pred_scores']
# #         roi_labels_dict[fidx] = preds[i]['pred_labels']


    

# max_preds = max(tensor.size(0) for tensor in roi_boxes_dict.values())
# # max_preds = 65
# # print(max_preds)
# past_boxes = []
# past_scores = []
# past_labels = []

# #the roi boxes structure has roi_cur_frame, roi_cur_frame-1, etc

# just_frames = target_frames_keys[3:][::-1]
# # just_frames = [frame_idx for frame_idx in target_frames]
# # just_frames.reverse()

# for frame_idx in just_frames:
#     preds = roi_boxes_dict[frame_idx]
#     scores = roi_scores_dict[frame_idx]
#     labels = roi_labels_dict[frame_idx]
#     num_preds = preds.size(0)
#     pad_size = max_preds - num_preds
#     padded_preds = F.pad(preds, (0, 0, 0, pad_size), mode = 'constant', value = 0)
#     padded_scores = F.pad(scores, (0, pad_size), mode = 'constant', value = 0)
#     padded_labels = F.pad(labels, (0, pad_size), mode = 'constant', value = 0)
#     past_boxes.append(padded_preds)
#     past_scores.append(padded_scores)
#     past_labels.append(padded_labels)
# past_boxes = torch.stack(past_boxes,dim = 0).detach().cpu().numpy()
# past_scores = torch.stack(past_scores, dim = 0).detach().cpu().numpy()
# past_labels = torch.stack(past_labels, dim = 0).detach().cpu().numpy()



# dict_mod_ptt = dataset[frame]
# dict_mod_ptt['roi_boxes'] = past_boxes
# dict_mod_ptt['roi_scores'] = past_scores
# dict_mod_ptt['roi_labels'] = past_labels
# # print(dict_mod_ptt['points'].shape)

# # add spoofed points to scene for single frame ptt. they might be used for something or the other
# if target_frames[frame]:
#     infos = dataset.infos
#     info_cur = infos[frame]
#     pose_cur = info_cur['pose'].reshape(4, 4) #ego pose of current frame


#     trace_pos_global_hom = np.hstack([trace_pos_global, np.ones((trace_pos_global.shape[0], 1))]) # put the global trace position into homogeneous coordinates
#     spf_cur = trace_pos_global_hom @ np.linalg.inv(pose_cur).T # get the coordinates of the spoofed points relative to base frame (lag T0) (this is in homo vector)
#     spf_cur_nolag = np.hstack([spf_cur[:, :3], trace['points'][:, -3:-1]]) # append 4th and 5th columns. do 6th column (lag) later

#     points = dataset[frame]['points']

#     # frame_points_spoof_rc = spoof_frame(spf_cur_nolag, points[:, :-1])
#     # zeros = np.zeros(frame_points_spoof_rc.shape[0])
#     # dict_mod_ptt['points'] = np.column_stack((frame_points_spoof_rc, zeros))
#     n_spoof_rc = []

#     frame_points_spoof_rc = spoof_frame(spf_cur_nolag, points, n_spoof_rc)
#     dict_mod_ptt['points'] = frame_points_spoof_rc



# # print(dict_mod_ptt['points'].shape)
# dict_mod_ptt = helpers_ptt.inject_gt_names(dict_mod_ptt, dataset.class_names)
# dict_mod_ptt = dataset.prepare_data(dict_mod_ptt)
# batch_mod_ptt = dataset.collate_batch([dict_mod_ptt])
# load_data_to_gpu(batch_mod_ptt)
# helpers_ptt.print_dict(batch_mod_ptt)
# model.eval()
# with torch.no_grad():
#     pred_ptt, _ = model.forward(batch_mod_ptt)
    
# # end_time = time.perf_counter()
# # execution_time = end_time - start_time
# # print(f"The code took {execution_time} seconds to execute.")

In [ ]:
# points = isolate_frame_points(dataset_mf[34]['points'], i)


In [59]:
# dataset_mf[34]['points'].shape

(126779, 6)

In [130]:
# target_frames_keys

[16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50]

In [131]:
# just_frames

[50,
 49,
 48,
 47,
 46,
 45,
 44,
 43,
 42,
 41,
 40,
 39,
 38,
 37,
 36,
 35,
 34,
 33,
 32,
 31,
 30,
 29,
 28,
 27,
 26,
 25,
 24,
 23,
 22,
 21,
 20,
 19]